# Bronze — Olist Marketing Funnel e IPCA

Executar no Jupyter da stack do grupo. Colocar os três CSV no diretório de trabalho do Jupyter ou adaptar os caminhos locais. Esta camada conserva todas as datas, os nomes e valores originais, acrescentando `_data_ingestao` e `_ficheiro_origem`. As tabelas são Delta e registadas na base `bronze` do Hive Metastore. Reexecução = substituição integral das três tabelas deste notebook.

In [1]:
from pathlib import Path
from hdfs import InsecureClient
FICHEIROS_LOCAIS = {
    "mql": "/home/jovyan/work/olist_marketing_qualified_leads_dataset.csv",
    "closed_deals": "/home/jovyan/work/olist_closed_deals_dataset.csv",
    "ipca": "/home/jovyan/work/IBGE_IPCA.csv"
}
HDFS_ORIGEM = "/projeto/bronze/bruno"
for nome, ficheiro in FICHEIROS_LOCAIS.items():
    assert Path(ficheiro).is_file(), f"Ficheiro não encontrado: {ficheiro}"
client = InsecureClient("http://hdfs-nn:9870", user="hdfs")
client.makedirs(HDFS_ORIGEM)
for nome, ficheiro in FICHEIROS_LOCAIS.items():
    destino = f"{HDFS_ORIGEM}/{Path(ficheiro).name}"
    client.delete(destino)
    client.upload(destino, ficheiro, overwrite=True)
    print(nome, "->", destino, client.status(destino)["length"], "bytes")

mql -> /projeto/bronze/bruno/olist_marketing_qualified_leads_dataset.csv 703836 bytes
closed_deals -> /projeto/bronze/bruno/olist_closed_deals_dataset.csv 171352 bytes
ipca -> /projeto/bronze/bruno/IBGE_IPCA.csv 5206 bytes


In [2]:
from pyspark.sql import SparkSession, functions as F
from delta import configure_spark_with_delta_pip
HDFS = "hdfs://hdfs-nn:9000"
WAREHOUSE = f"{HDFS}/warehouse"
builder = (SparkSession.builder.appName("EDSTD Bruno Marketing")
    .config("spark.sql.warehouse.dir", WAREHOUSE)
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0")
    .enableHiveSupport())
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)

Spark: 3.4.1


In [3]:
spark.sql("SHOW DATABASES").show()

spark.sql("""
    CREATE DATABASE IF NOT EXISTS bronze
    LOCATION 'hdfs://hdfs-nn:9000/warehouse/bronze.db'
""")

spark.sql("SHOW TABLES IN bronze").show()

+---------+
|namespace|
+---------+
|   bronze|
|  default|
|     demo|
|   silver|
+---------+

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|   bronze|         amazon_meta|      false|
|   bronze|      amazon_ratings|      false|
|   bronze| bronze_closed_deals|      false|
|   bronze|         bronze_ipca|      false|
|   bronze|          bronze_mql|      false|
|   bronze|      bronze_reviews|      false|
|   bronze|  bronze_rtr_reviews|      false|
|   bronze|  bronze_us_holidays|      false|
|   bronze| dataco_supply_chain|      false|
|   bronze|   datafiniti_precos|      false|
|   bronze|   inmet_observacoes|      false|
|   bronze|     olist_customers|      false|
|   bronze|   olist_order_items|      false|
|   bronze| olist_order_reviews|      false|
|   bronze|        olist_orders|      false|
|   bronze|olist_product_cat...|      false|
|   bronze|      olist_products|      false|
|  

In [4]:
spark.sql(
    "CREATE DATABASE IF NOT EXISTS bronze "
    "LOCATION 'hdfs://hdfs-nn:9000/warehouse/bronze.db'"
)
def ler_csv(nome):
    caminho = f"{HDFS}{HDFS_ORIGEM}/{Path(FICHEIROS_LOCAIS[nome]).name}"

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(caminho)
    )

    if nome == "ipca":
        df = df.withColumnRenamed(
            "IPCA Rate (last 12 months)",
            "ipca_rate_last_12_months"
        )

    return df
def gravar_bronze(nome, tabela):
    df = (ler_csv(nome)
          .withColumn("_ficheiro_origem", F.input_file_name())
          .withColumn("_data_ingestao", F.current_timestamp()))
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .option("path", f"{WAREHOUSE}/bronze.db/{tabela}/")
       .saveAsTable(f"bronze.{tabela}"))
    n = spark.table(f"bronze.{tabela}").count()
    print(f"bronze.{tabela}: {n} linhas")
    return n
contagens = {
    "mql": gravar_bronze("mql", "bronze_mql"),
    "closed_deals": gravar_bronze("closed_deals", "bronze_closed_deals"),
    "ipca": gravar_bronze("ipca", "bronze_ipca"),
}

bronze.bronze_mql: 8000 linhas
bronze.bronze_closed_deals: 842 linhas
bronze.bronze_ipca: 394 linhas


In [5]:
df_ipca = ler_csv("ipca")

print("Colunas originais:")
print(df_ipca.columns)

df_ipca.show(5, truncate=False)

Colunas originais:
['Date', 'ipca_rate_last_12_months']
+-------+------------------------+
|Date   |ipca_rate_last_12_months|
+-------+------------------------+
|01/1994|2693.84                 |
|02/1994|3035.71                 |
|03/1994|3417.39                 |
|04/1994|3828.49                 |
|05/1994|4331.19                 |
+-------+------------------------+
only showing top 5 rows



In [6]:
from pyspark.sql import functions as F

df_ipca = (
    ler_csv("ipca")
    .withColumnRenamed(
        "IPCA Rate (last 12 months)",
        "ipca_rate_last_12_months"
    )
    .withColumn("_ficheiro_origem", F.input_file_name())
    .withColumn("_data_ingestao", F.current_timestamp())
)

df_ipca.printSchema()
df_ipca.show(5, truncate=False)

root
 |-- Date: string (nullable = true)
 |-- ipca_rate_last_12_months: string (nullable = true)
 |-- _ficheiro_origem: string (nullable = false)
 |-- _data_ingestao: timestamp (nullable = false)

+-------+------------------------+------------------------------------------------------+-------------------------+
|Date   |ipca_rate_last_12_months|_ficheiro_origem                                      |_data_ingestao           |
+-------+------------------------+------------------------------------------------------+-------------------------+
|01/1994|2693.84                 |hdfs://hdfs-nn:9000/projeto/bronze/bruno/IBGE_IPCA.csv|2026-11-13 12:42:57.78656|
|02/1994|3035.71                 |hdfs://hdfs-nn:9000/projeto/bronze/bruno/IBGE_IPCA.csv|2026-11-13 12:42:57.78656|
|03/1994|3417.39                 |hdfs://hdfs-nn:9000/projeto/bronze/bruno/IBGE_IPCA.csv|2026-11-13 12:42:57.78656|
|04/1994|3828.49                 |hdfs://hdfs-nn:9000/projeto/bronze/bruno/IBGE_IPCA.csv|2026-11-13 12:42:5

In [7]:
(
    df_ipca.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", f"{WAREHOUSE}/bronze.db/bronze_ipca/")
    .saveAsTable("bronze.bronze_ipca")
)

print(
    "bronze.bronze_ipca:",
    spark.table("bronze.bronze_ipca").count(),
    "linhas"
)

bronze.bronze_ipca: 394 linhas


In [8]:
spark.sql("SHOW TABLES IN bronze").show()

for tabela in ["bronze_mql", "bronze_closed_deals", "bronze_ipca"]:
    n = spark.table(f"bronze.{tabela}").count()
    print(f"{tabela}: {n} linhas")

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|   bronze|         amazon_meta|      false|
|   bronze|      amazon_ratings|      false|
|   bronze| bronze_closed_deals|      false|
|   bronze|         bronze_ipca|      false|
|   bronze|          bronze_mql|      false|
|   bronze|      bronze_reviews|      false|
|   bronze|  bronze_rtr_reviews|      false|
|   bronze|  bronze_us_holidays|      false|
|   bronze| dataco_supply_chain|      false|
|   bronze|   datafiniti_precos|      false|
|   bronze|   inmet_observacoes|      false|
|   bronze|     olist_customers|      false|
|   bronze|   olist_order_items|      false|
|   bronze| olist_order_reviews|      false|
|   bronze|        olist_orders|      false|
|   bronze|olist_product_cat...|      false|
|   bronze|      olist_products|      false|
|   bronze|       olist_sellers|      false|
|   bronze|   superstore_orders|      false|
|   bronze

In [9]:
spark.sql("SHOW TABLES IN bronze").show()

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|   bronze|         amazon_meta|      false|
|   bronze|      amazon_ratings|      false|
|   bronze| bronze_closed_deals|      false|
|   bronze|         bronze_ipca|      false|
|   bronze|          bronze_mql|      false|
|   bronze|      bronze_reviews|      false|
|   bronze|  bronze_rtr_reviews|      false|
|   bronze|  bronze_us_holidays|      false|
|   bronze| dataco_supply_chain|      false|
|   bronze|   datafiniti_precos|      false|
|   bronze|   inmet_observacoes|      false|
|   bronze|     olist_customers|      false|
|   bronze|   olist_order_items|      false|
|   bronze| olist_order_reviews|      false|
|   bronze|        olist_orders|      false|
|   bronze|olist_product_cat...|      false|
|   bronze|      olist_products|      false|
|   bronze|       olist_sellers|      false|
|   bronze|   superstore_orders|      false|
|   bronze

## Validação — contagens anteriores apenas como referência

In [10]:
# O IPCA do Kaggle é atualizado todos os meses: a versão descarregada a 12/11/2026 tem 394 meses (01/1994 a 10/2026)
esperado = {"mql": 8000, "closed_deals": 842, "ipca": 394}
for nome, n in contagens.items():
    print(f"{nome}: {n} | referência: {esperado[nome]}")
    assert n == esperado[nome], f"Contagem diferente para {nome}; verificar origem"
for tabela in ("bronze_mql", "bronze_closed_deals", "bronze_ipca"):
    df = spark.table(f"bronze.{tabela}")
    assert {"_data_ingestao", "_ficheiro_origem"}.issubset(df.columns)
    assert df.filter(F.col("_data_ingestao").isNull() | F.col("_ficheiro_origem").isNull()).count() == 0
    print(tabela, df.columns)
spark.sql("SHOW TABLES IN bronze").show(truncate=False)

mql: 8000 | referência: 8000
closed_deals: 842 | referência: 842
ipca: 394 | referência: 394
bronze_mql ['mql_id', 'first_contact_date', 'landing_page_id', 'origin', '_ficheiro_origem', '_data_ingestao']
bronze_closed_deals ['mql_id', 'seller_id', 'sdr_id', 'sr_id', 'won_date', 'business_segment', 'lead_type', 'lead_behaviour_profile', 'has_company', 'has_gtin', 'average_stock', 'business_type', 'declared_product_catalog_size', 'declared_monthly_revenue', '_ficheiro_origem', '_data_ingestao']
bronze_ipca ['Date', 'ipca_rate_last_12_months', '_ficheiro_origem', '_data_ingestao']
+---------+---------------------------------------+-----------+
|namespace|tableName                              |isTemporary|
+---------+---------------------------------------+-----------+
|bronze   |amazon_meta                            |false      |
|bronze   |amazon_ratings                         |false      |
|bronze   |bronze_closed_deals                    |false      |
|bronze   |bronze_ipca         

In [11]:
spark.stop()